In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 日本語フォント設定
# plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_style("whitegrid")
plt.style.use("default")

In [ ]:
# 日本語フォント設定
matplotlib.rc("font", family="IPAexGothic")

In [ ]:
# 現在の最大表示列数の出力
pd.get_option("display.max_columns")

In [ ]:
# 最大表示列数の指定
pd.set_option("display.max_columns", 150)

In [ ]:
# 現在の最大表示行数の出力
pd.get_option("display.max_rows")

In [ ]:
# 最大表示行数の指定
pd.set_option("display.max_rows", 40)

In [ ]:
# レース結果情報を読み込み
# データの読み込み
results_df = pd.read_csv("data/results.csv")
results_df.head()

In [ ]:
# -------------------------------------------------------------------------
# データのフィルタリング
# -------------------------------------------------------------------------

In [ ]:
# -------------------------------------------------------------------------
# 1800mレースに絞り込み
# 順位がついたレースのみに絞り込み フライングや失格を除外
# -------------------------------------------------------------------------

In [ ]:
# 距離が1800mのレースデータにフィルタリング
results_df = results_df[results_df["距離"] == 1800]

In [ ]:
# 着順が "F", "S0", "S1", "S2", "L0", "L1", "K0", "K1" のデータはフィルタリング
results_df = results_df[
    ~results_df["着順"].isin(["F", "S0", "S1", "S2", "L0", "L1", "K0", "K1"])
]
# 着順を整数に変換
results_df["着順"] = results_df["着順"].astype(int)

In [ ]:
# -------------------------------------------------------------------------
# データの作成
# -------------------------------------------------------------------------

In [ ]:
# "年" "月" "日" 列から日付列を作成
# 年,月,日を結合して日付型に変換
results_df.insert(
    0,
    "開催日",
    pd.to_datetime(
        results_df["年"].astype(str)
        + "-"
        + results_df["月"].astype(str)
        + "-"
        + results_df["日"].astype(str)
    ),
)

In [ ]:
# "年" "月" "日" "レース場番号" "レース番号"からレースID列を作成
results_df.insert(
    1,
    "レースID",
    results_df["年"].astype(str)
    + results_df["月"].astype(str).str.zfill(2)
    + results_df["日"].astype(str).str.zfill(2)
    + results_df["レース場番号"].astype(str).str.zfill(2)
    + results_df["レース番号"].astype(str).str.zfill(2),
)

In [ ]:
# レースタイム 分.秒.ms 形式を秒に変換する関数
def time_to_seconds(time_str):
    try:
        minutes, seconds, milli100seconds = time_str.split(".")
        total_seconds = int(minutes) * 60 + float(seconds) + float(milli100seconds) / 10
        return total_seconds
    except:
        return np.nan


# レースタイム列を秒に変換
results_df["レースタイム秒"] = results_df["レースタイム"].apply(time_to_seconds)

In [ ]:
# -------------------------------------------------------------------------
# 不要カラムを削除
# -------------------------------------------------------------------------

In [ ]:
# "年" "月" "日" 列の削除
results_df = results_df.drop(columns=["年", "月", "日"])

In [ ]:
# レースタイム列を削除
results_df = results_df.drop(columns=["レースタイム"])

In [ ]:
# レースの配当情報などを削除
drop_columns = [
    "単勝_艇番",
    "単勝_払戻金",
    "複勝1着_艇番",
    "複勝1着_払戻金",
    "複勝2着_艇番",
    "複勝2着_払戻金",
    "2連単_艇番",
    "2連単_払戻金",
    "2連単_人気",
    "2連複_艇番",
    "2連複_払戻金",
    "2連複_人気",
    "拡連複1_艇番",
    "拡連複1_払戻金",
    "拡連複1_人気",
    "拡連複2_艇番",
    "拡連複2_払戻金",
    "拡連複2_人気",
    "拡連複3_艇番",
    "拡連複3_払戻金",
    "拡連複3_人気",
    "3連単_艇番",
    "3連単_払戻金",
    "3連単_人気",
    "3連複_艇番",
    "3連複_払戻金",
    "3連複_人気",
]

results_df = results_df.drop(columns=drop_columns)

In [ ]:
results_df.head(6)

In [ ]:
results_df.info()

In [ ]:
# 進入を整数に変換
results_df["進入"] = results_df["進入"].astype(int)

In [ ]:
# -------------------------------------------------------------------------
# 選手登番・艇番ごとの成績集計
# -------------------------------------------------------------------------

In [ ]:
# 選手登番・艇番ごとの成績集計
racers_lane_stats_df = (
    results_df.groupby(["選手登番", "艇番"])
    .agg(
        出走回数=("レースID", "count"),
        平均着順=("着順", "mean"),
        勝率=("着順", lambda x: (x == 1).sum() / len(x)),
        # 着順が1か2の回数を複勝率として集計
        複勝率=("着順", lambda x: ((x == 1) | (x == 2)).sum() / len(x)),
        二着率=("着順", lambda x: (x == 2).sum() / len(x)),
        三着率=("着順", lambda x: (x == 3).sum() / len(x)),
        四着率=("着順", lambda x: (x == 4).sum() / len(x)),
        五着率=("着順", lambda x: (x == 5).sum() / len(x)),
        六着率=("着順", lambda x: (x == 6).sum() / len(x)),
        平均レースタイム秒=("レースタイム秒", "mean"),
    )
    .reset_index()
)

In [ ]:
# -------------------------------------------------------------------------
# レース結果情報に選手登番・艇番ごとの成績をマージ
# -------------------------------------------------------------------------

In [ ]:
train_df = results_df[["レースID", "選手登番", "艇番", "着順"]].merge(
    racers_lane_stats_df[["勝率", "複勝率", "平均レースタイム秒", "選手登番", "艇番"]],
    on=["選手登番", "艇番"],
    how="left",
)

In [ ]:
# レースID内の艇番ごとの勝率差を計算して新しい特徴量を作成
train_df["勝率差"] = train_df.groupby("レースID")["勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの複勝率差を計算して新しい特徴量を作成
train_df["複勝率差"] = train_df.groupby("レースID")["複勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの平均レースタイム秒差を計算して新しい特徴量を作成
train_df["平均レースタイム秒差"] = train_df.groupby("レースID")[
    "平均レースタイム秒"
].transform(lambda x: x - x.mean())

In [ ]:
train_df.head(40)

In [ ]:
# 最後列に目的変数の1着フラグを追加
train_df["1着フラグ"] = (train_df["着順"] == 1).astype(int)

In [ ]:
# train_dfの相関行列を表示
correlation_matrix = train_df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("train_dfの相関行列")
plt.show()

In [ ]:
# -------------------------------------------------------------------------
# 学習データの不要行を削除
# -------------------------------------------------------------------------

In [ ]:
# 着順, 勝率, 複勝率, 平均レースタイム秒列を削除
train_df = train_df.drop(columns=["着順"])
train_df = train_df.drop(columns=["勝率", "複勝率", "平均レースタイム秒"])
# レースID  選手登番を削除
train_df = train_df.drop(columns=["レースID", "選手登番"])
train_df.head(40)

In [ ]:
# # 平均レースタイム秒差がNaNの行を削除
# train_df = train_df.dropna(subset=["平均レースタイム秒差"])

In [ ]:
# # 勝率差の列を削除
# train_df = train_df.drop(columns=["勝率差"])

In [ ]:
# CSVファイルとして保存
train_df.to_csv("data/train_data.csv", index=False)
# -------------------------------------------------------------------------
# 処理終了
# -------------------------------------------------------------------------

In [ ]:
# -------------------------------------------------------------------------
# 予測用のpred.csvデータを作成する
# -------------------------------------------------------------------------

In [ ]:
# 日付を指定してprogram.csvから予測用のpred.csvデータを作成する

In [ ]:
programs_df = pd.read_csv("data/programs.csv")
programs_df.head()

In [ ]:
# -------------------------------------------------------------------------
# 必要な列データの作成
# -------------------------------------------------------------------------

In [ ]:
# "年" "月" "日" 列から日付列を作成
# 年,月,日を結合して日付型に変換
programs_df.insert(
    0,
    "開催日",
    pd.to_datetime(
        programs_df["年"].astype(str)
        + "-"
        + programs_df["月"].astype(str)
        + "-"
        + programs_df["日"].astype(str)
    ),
)

In [ ]:
# "年" "月" "日" "レース場番号" "レース番号"からレースID列を作成して先頭に挿入
programs_df.insert(
    1,
    "レースID",
    programs_df["年"].astype(str)
    + programs_df["月"].astype(str).str.zfill(2)
    + programs_df["日"].astype(str).str.zfill(2)
    + programs_df["レース場番号"].astype(str).str.zfill(2)
    + programs_df["レース番号"].astype(str).str.zfill(2),
)

In [ ]:
# 枠番を艇番に変更
programs_df = programs_df.rename(columns={"枠番": "艇番"})

In [ ]:
# 不要列を削除
programs_df = programs_df.drop(columns=["年", "月", "日"])

In [ ]:
# -------------------------------------------------------------------------
# 指定日とレース場番号を指定して必要な列データの作成
# -------------------------------------------------------------------------

In [ ]:
pred_date = "2025-11-08"
pred_racecourse_number = 22

In [ ]:
programs_df = programs_df[
    (programs_df["開催日"] == pd.to_datetime(pred_date))
    & (programs_df["レース場番号"] == pred_racecourse_number)
]
programs_df.head()

In [ ]:
# 必要な列データの抽出して、コース別選手情報とマージする
pred_df = programs_df[["レースID", "選手登番", "艇番"]].merge(
    racers_lane_stats_df[["勝率", "複勝率", "平均レースタイム秒", "選手登番", "艇番"]],
    on=["選手登番", "艇番"],
    how="left",
)

In [ ]:
# レースID内の艇番ごとの勝率差を計算して新しい特徴量を作成
pred_df["勝率差"] = pred_df.groupby("レースID")["勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの複勝率差を計算して新しい特徴量を作成
pred_df["複勝率差"] = pred_df.groupby("レースID")["複勝率"].transform(
    lambda x: x - x.mean()
)

In [ ]:
# レースID内の艇番ごとの平均レースタイム秒差を計算して新しい特徴量を作成
pred_df["平均レースタイム秒差"] = pred_df.groupby("レースID")[
    "平均レースタイム秒"
].transform(lambda x: x - x.mean())

In [ ]:
# 不要カラムを削除
pred_df = pred_df.drop(columns=["勝率", "複勝率", "平均レースタイム秒"])

In [ ]:
pred_df.head(40)